# YOLOv8-OBB Training: Bottle vs Can Detection (v4)

**Dataset:** Roboflow `bouteille/bottle-and-can` v7, re-split for class balance  
**Model:** YOLOv8n-OBB  |  **GPU:** T4  |  **Epochs:** 100

## 1. Environment

In [ ]:
!nvidia-smi
%pip install -q ultralytics roboflow
import ultralytics; ultralytics.checks()

## 2. Download from Roboflow

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="l2vJEUtQX03ZwuWPpsXi")
project = rf.workspace("bouteille").project("bottle-and-can")
dataset = project.version(7).download("yolov8-obb", location="/content/raw")
print("Downloaded:", dataset.location)

## 3. Balanced Split
Roboflow v7 valid = 93% bottle / 7% can → causes model to detect everything as bottle.  
We re-split all images into balanced 80/10/10.

In [ ]:
import random, shutil, os
from collections import defaultdict, Counter
from pathlib import Path

random.seed(42)
SRC = Path("/content/raw")
DST = Path("/content/dataset")

def dominant_class(lbl_path):
    c = defaultdict(int)
    for line in lbl_path.read_text().strip().splitlines():
        if line: c[int(line.split()[0])] += 1
    return max(c, key=c.__getitem__) if c else None

by_class = defaultdict(list)
for split in ["train", "valid", "test"]:
    for img in (SRC / split / "images").glob("*.jpg"):
        lbl = SRC / split / "labels" / (img.stem + ".txt")
        if lbl.exists():
            cls = dominant_class(lbl)
            if cls is not None:
                by_class[cls].append((img, lbl))

n = min(len(v) for v in by_class.values())
balanced = []
for cls, pairs in by_class.items():
    random.shuffle(pairs)
    balanced.extend(pairs[:n])
random.shuffle(balanced)

total = len(balanced)
n_test = max(50, int(total * 0.10))
n_val  = max(50, int(total * 0.10))
n_train = total - n_test - n_val

splits_out = {
    "train": balanced[:n_train],
    "valid": balanced[n_train:n_train+n_val],
    "test":  balanced[n_train+n_val:],
}

if DST.exists(): shutil.rmtree(DST)

for split, pairs in splits_out.items():
    (DST/split/"images").mkdir(parents=True, exist_ok=True)
    (DST/split/"labels").mkdir(parents=True, exist_ok=True)
    for img, lbl in pairs:
        shutil.copy(img, DST/split/"images"/img.name)
        shutil.copy(lbl, DST/split/"labels"/lbl.name)

(DST/"data.yaml").write_text(
    "path: /content/dataset\ntrain: train/images\nval: valid/images\n"
    "test: test/images\n\nnames:\n  0: bottle\n  1: can\n\nnc: 2\n"
)

print(f"train:{n_train}  val:{n_val}  test:{n_test}")
for split in ["train", "valid", "test"]:
    c = Counter()
    for lbl in (DST/split/"labels").glob("*.txt"):
        for line in lbl.read_text().strip().splitlines():
            if line: c[int(line.split()[0])] += 1
    t = sum(c.values())
    print(f"  {split}: bottle={c[0]}({100*c[0]//max(t,1)}%) can={c[1]}({100*c[1]//max(t,1)}%)")

## 4. Training

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-obb.pt")
model.train(
    data="/content/dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    patience=20,
    mosaic=1.0,
    mixup=0.15,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=15.0, translate=0.1, scale=0.5,
    project="/content/runs",
    name="bottle_can_obb_v4",
    exist_ok=True,
    plots=True,
)

## 5. Evaluate

In [ ]:
best = "/content/runs/bottle_can_obb_v4/weights/best.pt"
model = YOLO(best)
metrics = model.val(data="/content/dataset/data.yaml", split="test", plots=True)
per_class = dict(zip(["bottle", "can"], metrics.box.maps))
print(f"mAP@0.5:       {metrics.box.map50:.4f}")
print(f"bottle mAP@0.5:{per_class['bottle']:.4f}")
print(f"can    mAP@0.5:{per_class['can']:.4f}")

## 6. Save to Google Drive

In [ ]:
from google.colab import drive, files
import shutil

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/yolo-portfolio"
os.makedirs(f"{DRIVE}/models", exist_ok=True)

shutil.copy(best, f"{DRIVE}/models/best.pt")
if os.path.exists(f"{DRIVE}/results/training"):
    shutil.rmtree(f"{DRIVE}/results/training")
shutil.copytree("/content/runs/bottle_can_obb_v4", f"{DRIVE}/results/training")
print("Saved to Drive.")

# Also download directly
files.download(best)